In [7]:
import copy
from collections import deque

# -------------------------------
# Helper Functions
# -------------------------------

# Returns all neighbors of a cell in Sudoku (row, column, and 3x3 box)
def get_neighbors(cell):
    row, col = cell
    neighbors = set()

    # Row and column constraints (all cells in same row/column)
    for i in range(9):
        neighbors.add((row, i))
        neighbors.add((i, col))

    # 3x3 subgrid constraints
    box_row, box_col = 3 * (row // 3), 3 * (col // 3)
    for i in range(box_row, box_row + 3):
        for j in range(box_col, box_col + 3):
            neighbors.add((i, j))

    # Remove the cell itself from neighbors
    neighbors.remove(cell)
    return neighbors


# -------------------------------
# AC-3 Algorithm
# -------------------------------

# AC-3 algorithm for enforcing arc consistency
def ac3(domains):
    queue = deque()

    # Initialize queue with all arcs
    for xi in domains:
        for xj in get_neighbors(xi):
            queue.append((xi, xj))

    # Process all arcs
    while queue:
        xi, xj = queue.popleft()

        # Revise domain of xi with respect to xj
        if revise(domains, xi, xj):
            # If domain becomes empty, no solution exists
            if len(domains[xi]) == 0:
                return False

            # Add affected neighbors back to queue
            for xk in get_neighbors(xi):
                if xk != xj:
                    queue.append((xk, xi))
    return True


# Removes inconsistent values from xi's domain
def revise(domains, xi, xj):
    revised = False

    for x in set(domains[xi]):
        # Constraint: neighboring cells cannot have same value
        if not any(x != y for y in domains[xj]):
            domains[xi].remove(x)
            revised = True

    return revised


# -------------------------------
# Backtracking + Forward Checking
# -------------------------------

# Global counters for performance tracking
backtrack_calls = 0
failures = 0


# Select variable using MRV (Minimum Remaining Values heuristic)
def select_unassigned_variable(domains):
    unassigned = [v for v in domains if len(domains[v]) > 1]
    return min(unassigned, key=lambda var: len(domains[var]))


# Check if assignment is complete
def is_complete(domains):
    return all(len(domains[v]) == 1 for v in domains)


# Backtracking search with forward checking and AC-3
def backtrack(domains):
    global backtrack_calls, failures
    backtrack_calls += 1

    # If all variables assigned, solution found
    if is_complete(domains):
        return domains

    # Select next variable (MRV heuristic)
    var = select_unassigned_variable(domains)

    # Try each possible value
    for value in sorted(domains[var]):  # numeric order
        new_domains = copy.deepcopy(domains)
        new_domains[var] = {value}

        # -------------------------------
        # Forward Checking
        # -------------------------------
        consistent = True
        for neighbor in get_neighbors(var):
            if value in new_domains[neighbor]:
                new_domains[neighbor].discard(value)
                if len(new_domains[neighbor]) == 0:
                    consistent = False
                    break

        # If consistent, apply AC-3
        if consistent and ac3(new_domains):
            result = backtrack(new_domains)
            if result:
                return result

    # If no value works, count failure
    failures += 1
    return None


# -------------------------------
# Input / Output Functions
# -------------------------------

# Reads Sudoku board from file
def read_board(filename):
    board = []
    with open(filename) as f:
        for line in f:
            board.append([int(x) for x in line.strip()])
    return board


# Initialize domains for each cell
def init_domains(board):
    domains = {}
    for i in range(9):
        for j in range(9):
            if board[i][j] == 0:
                domains[(i, j)] = set(range(1, 10))
            else:
                domains[(i, j)] = {board[i][j]}
    return domains


# Print final Sudoku board
def print_board(domains):
    for i in range(9):
        row = []
        for j in range(9):
            if len(domains[(i, j)]) == 1:
                row.append(str(next(iter(domains[(i, j)]))))
            else:
                row.append("0")
        print(" ".join(row))


# -------------------------------
# Solver Function
# -------------------------------

def solve(filename):
    global backtrack_calls, failures
    backtrack_calls = 0
    failures = 0

    print(f"\nSolving: {filename}")

    # Load board
    board = read_board(filename)
    domains = init_domains(board)

    # Initial AC-3 preprocessing
    if not ac3(domains):
        print("No solution exists.")
        return

    # Run backtracking search
    result = backtrack(domains)

    if result:
        print("\nSolved Board:")
        print_board(result)
    else:
        print("No solution found.")

    # Print statistics
    print("\nBacktrack calls:", backtrack_calls)
    print("Failures:", failures)


# -------------------------------
# Run All Boards
# -------------------------------

if __name__ == "__main__":
    solve("easy.txt")
    solve("medium.txt")
    solve("hard.txt")
    solve("veryhard.txt")


Solving: easy.txt

Solved Board:
7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3

Backtrack calls: 1
Failures: 0

Solving: medium.txt

Solved Board:
8 7 5 9 3 6 1 4 2
1 6 9 7 2 4 3 8 5
2 4 3 8 5 1 6 7 9
4 5 2 6 9 7 8 3 1
9 8 6 4 1 3 2 5 7
7 3 1 5 8 2 9 6 4
5 1 7 3 6 9 4 2 8
6 2 8 1 4 5 7 9 3
3 9 4 2 7 8 5 1 6

Backtrack calls: 2
Failures: 0

Solving: hard.txt

Solved Board:
1 5 2 3 4 6 8 9 7
4 3 7 1 8 9 6 5 2
6 8 9 5 7 2 3 1 4
8 2 1 6 3 7 9 4 5
5 4 3 8 9 1 7 2 6
9 7 6 4 2 5 1 8 3
7 9 8 2 5 3 4 6 1
3 6 5 9 1 4 2 7 8
2 1 4 7 6 8 5 3 9

Backtrack calls: 7
Failures: 2

Solving: veryhard.txt

Solved Board:
4 3 1 8 6 7 9 2 5
6 5 2 4 9 1 3 8 7
8 9 7 5 3 2 1 6 4
3 8 4 9 7 6 5 1 2
5 1 9 2 8 4 7 3 6
2 7 6 3 1 5 8 4 9
9 4 3 7 2 8 6 5 1
7 6 5 1 4 3 2 9 8
1 2 8 6 5 9 4 7 3

Backtrack calls: 56
Failures: 43


## Final Verdicts

For the **easy** board, the solver completes the puzzle almost instantly with only **1 backtrack call** and **no failures**, showing that constraints are strong enough for direct deduction.  

In the **medium** board, a slight increase to **2 backtrack calls** still indicates a mostly straightforward search with no wrong paths.  

The **hard** board shows more complexity with **7 backtracks and 2 failures**, meaning the solver had to explore and recover from incorrect choices.  

For the **very hard** board, **56 backtracks and 43 failures** reflect a highly complex search space where many assignments lead to dead ends, requiring extensive backtracking and pruning.  


As the difficulty of the Sudoku puzzles increases, the number of guesses required during the search also increases. Forward checking helps by reducing invalid options early, but it cannot fully prevent the need for backtracking. The AC-3 algorithm improves efficiency by enforcing consistency and reducing variable domains before and during the search. Despite this, harder puzzles still require significant exploration of possible values. This leads to a noticeable rise in both backtracking steps and failures. Overall, the computational effort grows rapidly as puzzle complexity increases.